# One-hot encoding

How well do the classifiers detect enzymes with one-hot sequence features? [Random forest, logistic regression and SVM results](../../results/feature_comparison/gridsearch_model_comparison/one_hot_encoding/)


In [ ]:
import sys

sys.path.append('..')

In [ ]:
import numpy as np
from src.preprocessing import batch_encode_and_flatten
import pandas as pd
from pathlib import Path
from src.utils import calculate_search_grid, fasta_to_tsv

# Import dataset / sequences
#df_path = Path('../data/preprocessed/exp_proteins_c100_flens25-75_usamp_enz.tsv')
train_path = Path('../data/split/exp_proteins_c30_flens2-98_enz_train.tsv')
test_path = Path('../data/split/exp_proteins_c30_flens2-98_enz_test.tsv')
#df_path = Path('../data/preprocessed_new/exp_proteins_c30_flens2-98_usamp_enz.tsv')
df_train = pd.read_csv(train_path, sep='\t')
df_test = pd.read_csv(test_path, sep='\t')

In [ ]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix, make_scorer

train_list = list(df_train['Sequence'])
test_list = list(df_test['Sequence'])
max_len = max(max(len(s) for s in train_list), max(len(s) for s in test_list))

# One-hot encode the sequences
X_train = batch_encode_and_flatten(seq_list=train_list, max_len=max_len)
X_test = batch_encode_and_flatten(seq_list=test_list, max_len=max_len)

# Define labels
y_train = list(df_train['Enzyme'])
y_test = list(df_test['Enzyme'])

In [ ]:
# Using simple train-test split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=45)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import LinearSVC

# Initializing the estimators
rf = RandomForestClassifier()
logr = LogisticRegression(max_iter=1000)
svm = LinearSVC(max_iter=5000)

mcc_scorer = make_scorer(matthews_corrcoef)

In [ ]:
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
logr.fit(X_train, y_train)
y_pred = logr.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print(matthews_corrcoef(y_test, y_pred))

In [ ]:
# Random Forest
n_iter = 10
param_dist_rf = {'max_depth': [None, 10, 20, 30], 'max_features': ['sqrt', 'log2'], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2, 4]}
rand_grid_rf = calculate_search_grid(rf, X_train, y_train, X_test, y_test, n_iter, param_dist_rf)
rand_grid_rf.to_csv('../results/perfomance_eval/gridsearch_model_comparison/one_hot_encoding/one_hot_encoding_rf.tsv', sep='\t', index=False)


In [ ]:
# Logistic Regression
n_iter = 10
param_dist_logr = {'max_iter': [1000], 'C': [0.0001, 0.001, 0.01], 'tol': [0.0001, 0.001, 0.01]}
rand_grid_logr = calculate_search_grid(logr, X_train, y_train, X_test, y_test, n_iter, param_dist_logr, randomized=False)
rand_grid_logr.to_csv('../results/performance_eval/gridsearch_model_comparison/one_hot_encoding/one_hot_encoding_logr.tsv', sep='\t', index=False)

In [ ]:
# SVM
n_iter = 10
param_dist_svm = {'loss': ['hinge', 'squared_hinge'], 'C': [0.1, 1, 10], 'fit_intercept': [True, False]}
rand_grid_svm = calculate_search_grid(svm, X_train, y_train, X_test, y_test, n_iter, param_dist_svm)
rand_grid_svm.to_csv('../results/performance_eval/gridsearch_model_comparison/one_hot_encoding/one_hot_encoding_svm.tsv', sep='\t', index=False)